# Basline classifier using 1D CNN for time series data

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn as nn
import copy
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
PROCESSED_DIR = PROJECT_ROOT / 'dataset' / 'processed'
MODEL_DIR = PROJECT_ROOT / "models"
data = np.load(PROCESSED_DIR / 'splits.npz')

X_train = data['X_train']
X_val   = data['X_val']
X_test  = data['X_test']
y_train = data['y_train']
y_val   = data['y_val']
y_test  = data['y_test']

print('Train:', X_train.shape, 'labels:', y_train.shape)
print('Val  :', X_val.shape)
print('Test :', X_test.shape)

Train: (14525, 6, 300) labels: (14525,)
Val  : (2075, 6, 300)
Test : (4150, 6, 300)


In [3]:
class HARDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]


def make_loader(X, y=None, batch_size=128, shuffle=False):
    ds = HARDataset(X, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

train_loader = make_loader(X_train, y_train, batch_size=128, shuffle=True)
val_loader   = make_loader(X_val, y_val, batch_size=128)
test_loader  = make_loader(X_test, y_test, batch_size=128)

In [4]:
X, y = next(iter(train_loader))

print(f"X shape: {X.shape}\ny shape: {y.shape}")
print("y unique:", torch.unique(y))
print(f"X mean: {X.mean():.4f}\nX std: {X.std():.4f}")
print(f"X_train mean: {X_train.mean():.4f}\nX_train std: {X_train.std():.4f}")

X shape: torch.Size([128, 6, 300])
y shape: torch.Size([128])
y unique: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])
X mean: -0.0000
X std: 0.9999
X_train mean: -0.0000
X_train std: 0.9999


In [5]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=18):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(6, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(3),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 25, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
def train_classifier(model, train_loader, val_loader, epochs=15, lr=1e-3, device="cpu"):
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                preds = out.argmax(dim=1)

                correct += (preds == y).sum().item()
                total += y.size(0)

        acc = correct / total
        avg_loss = total_loss / len(train_loader)
        
        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model

In [6]:
device="cpu"
model = BaselineCNN()
model = train_classifier(
    model,
    train_loader,
    val_loader,
    epochs=15,
    lr=5e-4,
    device=device
)
torch.save(model.state_dict(), MODEL_DIR / 'baseline_classifier.pth')

Epoch 1 | Loss: 1.8345 | Val Acc: 0.4973
Epoch 2 | Loss: 1.1449 | Val Acc: 0.6284
Epoch 3 | Loss: 0.8767 | Val Acc: 0.6757
Epoch 4 | Loss: 0.7488 | Val Acc: 0.6858
Epoch 5 | Loss: 0.6614 | Val Acc: 0.7195
Epoch 6 | Loss: 0.6004 | Val Acc: 0.7166
Epoch 7 | Loss: 0.5671 | Val Acc: 0.7508
Epoch 8 | Loss: 0.5315 | Val Acc: 0.7643
Epoch 9 | Loss: 0.4966 | Val Acc: 0.7648
Epoch 10 | Loss: 0.4717 | Val Acc: 0.7769
Epoch 11 | Loss: 0.4444 | Val Acc: 0.8159
Epoch 12 | Loss: 0.4252 | Val Acc: 0.8169
Epoch 13 | Loss: 0.4123 | Val Acc: 0.8188
Epoch 14 | Loss: 0.3926 | Val Acc: 0.7817
Epoch 15 | Loss: 0.3690 | Val Acc: 0.8289


## Evaluation

In [7]:
def evaluate(model, loader, device="cpu"):
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            out = model(X)
            preds = out.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')

    return acc, f1

In [8]:
model = BaselineCNN().to(device="cpu")
model.load_state_dict(torch.load(MODEL_DIR / "baseline_classifier.pth"))
model.eval()

test_acc, test_f1 = evaluate(model, test_loader)
print("Test Accuracy:", test_acc)
print("Test Macro F1:", test_f1)

Test Accuracy: 0.8272289156626506
Test Macro F1: 0.8280014071127177


In [9]:
def full_report(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            out = model(X)
            preds = out.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    print(classification_report(all_labels, all_preds))

full_report(model, test_loader)

              precision    recall  f1-score   support

           0       0.49      0.89      0.63       377
           1       0.74      0.29      0.42       375
           2       0.84      0.94      0.89       359
           3       0.88      0.96      0.92       373
           4       0.95      1.00      0.97       436
           5       0.95      0.67      0.78       363
           6       0.88      0.97      0.92       352
           7       0.99      0.79      0.88       267
           8       0.95      0.96      0.96       133
           9       0.97      0.94      0.95        96
          10       0.92      0.94      0.93       201
          11       0.79      0.95      0.86       176
          12       0.95      0.92      0.94        63
          13       0.66      0.40      0.50        52
          14       1.00      0.77      0.87       119
          15       0.80      0.89      0.85       160
          16       0.90      0.61      0.73       156
          17       0.95    

## 20% labeled data

In [10]:
def sample_labels(X, y, fraction, random_state=42):
    if fraction >= 1.0:
        return X, y
    _, X_f, _, y_f = train_test_split(X, y, test_size=fraction, stratify=y, random_state=random_state)
    return X_f, y_f

In [11]:
X_small, y_small = sample_labels(X_train, y_train, fraction=0.2)
train_loader_small = make_loader(X_small, y_small, batch_size=128, shuffle=True)

classifier_small = BaselineCNN()
model_small = train_classifier(
    classifier_small,
    train_loader_small,
    val_loader,
    epochs=15,
    lr=5e-4,
    device=device
)

torch.save(model_small.state_dict(), MODEL_DIR / 'baseline_20.pth')

Epoch 1 | Loss: 2.3140 | Val Acc: 0.2612
Epoch 2 | Loss: 1.9507 | Val Acc: 0.3470
Epoch 3 | Loss: 1.7007 | Val Acc: 0.3913
Epoch 4 | Loss: 1.5061 | Val Acc: 0.4120
Epoch 5 | Loss: 1.3170 | Val Acc: 0.4646
Epoch 6 | Loss: 1.1682 | Val Acc: 0.4593
Epoch 7 | Loss: 1.0200 | Val Acc: 0.5499
Epoch 8 | Loss: 0.9508 | Val Acc: 0.5701
Epoch 9 | Loss: 0.8997 | Val Acc: 0.5860
Epoch 10 | Loss: 0.8272 | Val Acc: 0.6010
Epoch 11 | Loss: 0.7562 | Val Acc: 0.6048
Epoch 12 | Loss: 0.7146 | Val Acc: 0.6352
Epoch 13 | Loss: 0.6708 | Val Acc: 0.6381
Epoch 14 | Loss: 0.6419 | Val Acc: 0.6704
Epoch 15 | Loss: 0.6054 | Val Acc: 0.6535


In [12]:
# Evaluate
model_small = BaselineCNN().to(device)
model_small.load_state_dict(torch.load(MODEL_DIR / "baseline_20.pth"))

test_acc, test_f1 = evaluate(model_small, test_loader)

print("Baseline 20% -> Test Acc:", test_acc)
print("Baseline 20% -> Test F1 :", test_f1)

Baseline 20% -> Test Acc: 0.68
Baseline 20% -> Test F1 : 0.6052804691347365
